In [1]:
!pip install faiss_cpu sentence-transformers transformers torch numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.1 MB/s eta 0:00:00:00:0100:01


# EMBEDDING

In [2]:
import json
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# -------- CONFIG --------
CHUNKS_PATH = "/kaggle/input/datasets/arjunsreenivasiiitk/justly/knowledge_chunks2.json"
INDEX_PATH = "e5_large_index.faiss"
META_PATH = "e5_large_index.pkl"
MODEL_NAME = "intfloat/e5-large-v2"
BATCH_SIZE = 64
# ------------------------

print("Loading chunks...")
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts = []
metadatas = []

for c in chunks:
    # Include metadata in embedding context
    meta_str = " | ".join(f"{k}: {v}" for k, v in c["metadata"].items())
    combined = f"{c['text']} || {meta_str}"
    texts.append(combined)
    metadatas.append(c)

print(f"Loaded {len(texts)} chunks")

print("Loading embedding model...")
model = SentenceTransformer(MODEL_NAME)

print("Embedding...")
embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.array(embeddings).astype("float32")

print("Building FAISS index...")
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

faiss.write_index(index, INDEX_PATH)

with open(META_PATH, "wb") as f:
    pickle.dump(metadatas, f)

print("Index built successfully!")

Loading chunks...
Loaded 29975 chunks
Loading embedding model...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

... (6 outputs left out)

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Embedding...


Batches:   0%|          | 0/469 [00:00<?, ?it/s]

Building FAISS index...
Index built successfully!


# INITIATION

In [1]:
!pip install faiss-cpu sentence-transformers transformers torch numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.7 MB/s eta 0:00:00:00:0100:01


In [2]:
import faiss
import json
import pickle
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---------- CONFIG ----------
INDEX_PATH = "/kaggle/input/datasets/arjunsreenivasiiitk/justly/index3.faiss"
META_PATH = "/kaggle/input/datasets/arjunsreenivasiiitk/justly/index3.pkl"
EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"
TOP_K = 6
# ----------------------------

print("Loading index...")
index = faiss.read_index(INDEX_PATH)

with open(META_PATH, "rb") as f:
    metadatas = pickle.load(f)

print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)


Loading index...


RuntimeError: Error in faiss::FileIOReader::FileIOReader(const char*) at /project/third-party/faiss/faiss/impl/io.cpp:69: Error: 'f' failed: could not open /kaggle/input/datasets/arjunsreenivasiiitk/justly/index3.faiss for reading: No such file or directory

In [4]:
def query_ui(tokenizer,model):
    while True:
        q = input("\nAsk a question (or 'exit'): ")
        if q.lower() == "exit":
            break
        print("\n", ask(q,tokenizer,model))

In [ ]:

def retrieve(query, k=TOP_K):
    q_emb = embedder.encode(
        query,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(np.array([q_emb]), k)

    results = []
    for idx in indices[0]:
        results.append(metadatas[idx])

    return results

def clean_text(text):
    bad_tokens = ["Question:", "Answer:", "Q:", "A:"]
    for t in bad_tokens:
        text = text.replace(t, "")
    return text.strip()


def build_context(docs, max_chars=3000):
    lines = []
    total = 0
    for d in docs:
        chunk = d["text"]
        if total + len(chunk) > max_chars:
            break
        meta = ", ".join(f"{k}: {v}" for k, v in d["metadata"].items())
        lines.append(f"- {chunk} ({meta})")
        total += len(chunk)
    return "\n".join(lines)


def ask(query,tokenizer,model):
    # print("retrieve started")
    docs = retrieve(query)
    # print("retrieve ended")

    # print("context started")
    context = build_context(docs)
    # print("context ended")

    prompt = f"""
      You are an expert Ayurvedic biomedical knowledge assistant.

      Rules:
      - Use ONLY the context
      - If information is missing, say "Not available in context"
      - Answer as a complete paragraph of all items found in context.
      - Do not stop early.
      - If the user asks for a list, give output in a list.



      Context:
        {context}

        Question:
        {query}

        Answer:
      """


    # print("tokenization started")
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    # print("tokenization ended")

    # print("generation started")
    output = model.generate(
      **inputs,
      max_new_tokens=300,     # ↓ from 300
      temperature=0.2,
      do_sample=False,        # IMPORTANT
      eos_token_id=tokenizer.convert_tokens_to_ids("<|endoftext|>"),
      pad_token_id=tokenizer.eos_token_id,
      use_cache=True
    )

    # print("generation ended")

    # 🔑 decode ONLY new tokens
    generated = output[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)

    if "Context:" in text:
        text = text.split("Context:")[0]
    return text.strip()



# EVALUATION

In [ ]:
!pip install ragas

In [ ]:
def handle_query(query,tokenizer,model):
    # print("retrieve started")
    docs = retrieve(query)
    # print("retrieve ended")
    context = build_context(docs)
    # print("context ended")

    prompt = f"""
      You are an expert Ayurvedic biomedical knowledge assistant.

      Rules:
      - Use ONLY the context
      - If information is missing, say "Not available in context"
      - Answer as a complete paragraph of all items found in context.
      - Do not stop early.
      - If the user asks for a list, give output in a list.



      Context:
        {context}

        Question:
        {query}

        Answer:
      """


    # print("tokenization started")
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    # print("tokenization ended")

    # print("generation started")
    output = model.generate(
      **inputs,
      max_new_tokens=300,     # ↓ from 300
      temperature=0.2,
      do_sample=False,        # IMPORTANT
      eos_token_id=tokenizer.eos_token_id,
      pad_token_id=tokenizer.eos_token_id,
      use_cache=True
    )

    # print("generation ended")

    # 🔑 decode ONLY new tokens
    generated = output[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)

    if "Context:" in text:
        text = text.split("Context:")[0]
    return text.strip(),context

In [ ]:
from datasets import Dataset
import json
import pandas as pd

def test_data_creation(model,tokenizer):
    with open("/kaggle/input/datasets/arjunsreenivasiiitk/justly/test_questions.json","r") as f:
        test_questions=json.load(f)
    
    evaluation_rows = []
    for i,q in enumerate(test_questions):
        print(f"{i+1}/50")
        answer, context = handle_query(
            q["question"],
            tokenizer,
            model,
        )
    
        evaluation_rows.append({
            "question": q["question"],
            "ground_truth": q["answer"],
            "answer": answer,
            "contexts": context.split("\n")
        })

    df = pd.DataFrame(evaluation_rows)
    dataset = Dataset.from_pandas(df)
    return dataset

In [ ]:
!pip install langchain

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    context_recall,
    context_precision,
    faithfulness,
    answer_relevancy
)
from langchain_community.llms import HuggingFacePipeline
from ragas.llms import LangchainLLMWrapper
from transformers import pipeline
import os
os.environ["OPENAI_API_KEY"] = "give-your-key"

def evaluation(dataset):
    result = evaluate(
        dataset,
        metrics=[
            context_recall,
            context_precision,
            faithfulness,
            answer_relevancy
        ],
    )
    print(result)

In [ ]:
LLM_MODEL = "microsoft/phi-3-mini-4k-instruct"
print("Loading LLM...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    dtype=torch.float16,
    device_map="auto",
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)

In [ ]:
dataset = test_data_creation(model,tokenizer)
dataset[:5]

In [ ]:
def fix_format(example):
    return {
        "question": example["question"],
        "ground_truth": example["ground_truth"],
        "answer": example["answer"],
        "retrieved_contexts": example["contexts"].split("\n")  # 🔑 FIX
    }

dataset = dataset.map(fix_format)

In [ ]:
# temp_dataset = dataset.select(range(20))
evaluation(dataset)

# NEW EVALUATION

In [1]:
!pip install faiss-cpu sentence-transformers transformers torch numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.1 MB/s eta 0:00:00:00:0100:01


In [2]:
import faiss
import json
import pickle
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import Dataset
import pandas as pd
from tqdm import tqdm

In [3]:
# ---------- CONFIG ----------
INDEX_PATH = "/kaggle/input/datasets/arjunsreenivasiiitk/justly/e5_large_index.faiss"
META_PATH = "/kaggle/input/datasets/arjunsreenivasiiitk/justly/e5_large_index.pkl"
EMBED_MODEL = "intfloat/e5-large-v2"
TOP_K = 10
# ----------------------------

print("Loading index...")
index = faiss.read_index(INDEX_PATH)

with open(META_PATH, "rb") as f:
    metadatas = pickle.load(f)

print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)


Loading index...


Loading embedding model...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [4]:
def retrieve(query, k=TOP_K):
    q_emb = embedder.encode(
        query,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(np.array([q_emb]), k)

    results = []
    for idx in indices[0]:
        results.append(metadatas[idx])

    return results

def clean_text(text):
    bad_tokens = ["Question:", "Answer:", "Q:", "A:"]
    for t in bad_tokens:
        text = text.replace(t, "")
    return text.strip()


def build_context(docs, max_chars=3000):
    lines = []
    total = 0
    for d in docs:
        chunk = d["text"]
        if total + len(chunk) > max_chars:
            break
        meta = ", ".join(f"{k}: {v}" for k, v in d["metadata"].items())
        lines.append(f"- {chunk} ({meta})")
        total += len(chunk)
    return "\n".join(lines)

def handle_query(query,tokenizer,model):
    docs = retrieve(query)
    context = build_context(docs)

    prompt = f"""
      You are an expert Ayurvedic biomedical knowledge assistant.

      Rules:
      - Use ONLY the context
      - If information is missing, say "Not available in context"
      - Answer as a complete paragraph of all items found in context.
      - Do not stop early.
      - If the user asks for a list, give output in a list.
      - Combine all the context text and give answer in a simple paragraph.



      Context:
        {context}

        Question:
        {query}

        Answer:
      """


    # print("tokenization started")
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    # print("tokenization ended")

    # print("generation started")
    output = model.generate(
      **inputs,
      max_new_tokens=300,     # ↓ from 300
      temperature=0.2,
      do_sample=False,        # IMPORTANT
      eos_token_id=tokenizer.eos_token_id,
      pad_token_id=tokenizer.eos_token_id,
      use_cache=True
    )

    # print("generation ended")

    # 🔑 decode ONLY new tokens
    generated = output[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)

    if "Context:" in text:
        text = text.split("Context:")[0]
    return text.strip(),context

In [5]:
def test_data_creation(model, tokenizer):
    with open("/kaggle/input/datasets/arjunsreenivasiiitk/justly/test_questions.json", "r") as f:
        test_questions = json.load(f)
    
    evaluation_rows = []

    for i, q in enumerate(tqdm(test_questions, desc="Processing", unit="question")):
        answer, context = handle_query(
            q["question"],
            tokenizer,
            model,
        )
    
        evaluation_rows.append({
            "question": q["question"],
            "ground_truth": q["answer"],
            "answer": answer,
            "contexts": context.split("\n"),
            "source": q['source']
        })

    df = pd.DataFrame(evaluation_rows)
    dataset = Dataset.from_pandas(df)
    return dataset

In [6]:
from sklearn.metrics.pairwise import cosine_similarity
import re
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

def normalize_text(text):
    text = text.lower()
    # Remove punctuation
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # Tokenize
    words = text.split()
    # Lemmatize (handles plurals, verb forms)
    words = [lemmatizer.lemmatize(w) for w in words]
    return set(words)

def is_grounded(expected, evidence_text, threshold=0.8):
    expected_words = normalize_text(expected)
    evidence_words = normalize_text(evidence_text)
    overlap = expected_words & evidence_words
    score = len(overlap) / len(expected_words)
    return score >= threshold, score

def fast_evaluate(expected,answer,source,evidence_chunks):
    expected = expected.lower()
    
    # 2. Retrieval correctness
    sources = [
        c.get("metadata", {}).get("source")
        for c in evidence_chunks
        if c.get("metadata", {}).get("source") is not None
    ]
    retrieval_correct = source in sources

    # 3. Grounding check (no hallucination)
    evidence_text = " ".join([c["text"].lower() for c in evidence_chunks])
    grounded,score = is_grounded(expected,evidence_text)
    
    return {
        "retrieval_correctness": retrieval_correct,
        "grounding_check":{
            "grounded": grounded,
            "grounded_score":score
        }
    }

def recall_at_k(question, expected_source, k=10):
    evidence = retrieve(
        question,
        k
    )

    sources = [
        c.get("metadata", {}).get("source")
        for c in evidence
        if c.get("metadata", {}).get("source") is not None
    ]
    return expected_source in sources

def semantic_score(expected, answer):
    e1 = embedder.encode([expected])
    e2 = embedder.encode([answer])
    
    return cosine_similarity(e1, e2)[0][0]

def precision_at_k(evidence_chunks, expected_source):
    sources = [
        c.get("metadata", {}).get("source")
        for c in evidence_chunks
        if c.get("metadata", {}).get("source") is not None
    ]
    relevant = sum(1 for s in sources if s == expected_source)
    return relevant / len(sources)

def extract_entities(text):
    # simple heuristic: uppercase words/numbers (HIF1A, VEGFA, etc.)
    return set(re.findall(r'\b[A-Z0-9]{3,}\b', text))

def completeness_score(expected, answer):
    expected_words = normalize_text(expected)
    answer_words = normalize_text(answer)

    return len(expected_words & answer_words) / len(expected_words)

def hallucination_score(answer, evidence_chunks):
    evidence_text = " ".join([c["text"].lower() for c in evidence_chunks])
    
    answer_words = normalize_text(answer)
    evidence_words = normalize_text(evidence_text)

    extra = answer_words - evidence_words
    
    return 1 - (len(extra) / max(len(answer_words), 1))

stopwords = {
    "the", "is", "in", "and", "of", "to", "a", "from", "with", "by",
    "are", "was", "were", "be", "this", "that", "it", "as", "at", "on",
    "for", "an", "or", "not", "but", "if", "then", "so", "than", "too",
    "very", "can", "will", "just", "into", "about", "over", "after",
    "before", "between", "through", "during", "above", "below", "up",
    "down", "out", "off", "again", "further", "once", "here", "there",
    "when", "where", "why", "how", "all", "any", "both", "each", "few",
    "more", "most", "other", "some", "such", "no", "nor", "only", "own",
    "same", "than", "too", "s", "t", "don", "should", "now", "d", "ll",
    "m", "o", "re", "ve", "y", "ain", "aren", "couldn", "didn", "doesn",
    "hadn", "hasn", "haven", "isn", "ma", "mightn", "mustn", "needn",
    "shan", "shouldn", "wasn", "weren", "won", "wouldn"
}

def extract_keywords(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    words = text.split()
    keywords = []
    
    for w in words:
        if w in stopwords:
            continue
        # keep important-looking tokens:
        # 1. longer words
        # 2. alphanumeric biomedical terms (hif1a, vegfa)
        if len(w) > 3 or re.search(r'\d', w):
            keywords.append(w)
    return set(keywords)

def weighted_keyword_score(expected, answer):
    k1 = extract_keywords(expected)
    k2 = extract_keywords(answer)

    if len(k1) == 0:
        return 1.0

    score = 0
    total_weight = 0
    for word in k1:
        # give higher weight to biomedical entities
        if re.search(r'\d', word): weight = 2
        else: weight = 1
        total_weight += weight
        if word in k2: score += weight

    return score / total_weight

def final_score(metrics):
    return (
        0.22 * metrics["semantic_score"] +
        0.22 * metrics["recall_at_k"] +
        0.18 * metrics["grounding_check"]["grounded_score"] +
        0.18 * metrics["keyword_score"] +
        0.10 * metrics["precision_at_k"] +
        0.10 * metrics["hallucination_score"]
    )
    
def evaluation_per_sample(sample, k=10):
    evidence_chunks = retrieve(sample['question'], k)

    expected = sample['ground_truth']
    answer = sample['answer']
    source = sample['source']
    question = sample['question']

    rule_based = fast_evaluate(expected, answer, source, evidence_chunks)
    recall_atk = recall_at_k(question, source)
    semscore = semantic_score(expected, answer)
    kw_score = weighted_keyword_score(expected, answer)
    precision = precision_at_k(evidence_chunks, source)
    hallucination = hallucination_score(answer, evidence_chunks)

    res = {}
    res.update(rule_based)

    res.update({
        "recall_at_k": recall_atk,
        "precision_at_k": precision,
        "semantic_score": semscore,
        "hallucination_score": hallucination,
        "keyword_score": kw_score
    })

    res["final_score"] = final_score(res)

    return res

In [7]:
def aggregate_results(results):

    metrics = {
        "semantic_score": [],
        "grounded_score": [],
        "recall_at_k": [],
        "precision_at_k": [],
        "keyword_score": [],
        "hallucination_score": [],
        "final_score": []
    }

    for r in results:
        metrics["semantic_score"].append(r["semantic_score"])
        metrics["grounded_score"].append(r["grounding_check"]["grounded_score"])
        metrics["recall_at_k"].append(int(r["recall_at_k"]))
        metrics["precision_at_k"].append(r["precision_at_k"])
        metrics["keyword_score"].append(r["keyword_score"])
        metrics["hallucination_score"].append(r["hallucination_score"])
        metrics["final_score"].append(r["final_score"])

    summary = {
        "avg_semantic_score": np.mean(metrics["semantic_score"]),
        "avg_grounding_score": np.mean(metrics["grounded_score"]),
        "recall_at_k": np.mean(metrics["recall_at_k"]),
        "avg_precision_at_k": np.mean(metrics["precision_at_k"]),
        "avg_keyword_score": np.mean(metrics["keyword_score"]),
        "avg_hallucination_score": np.mean(metrics["hallucination_score"]),
        "avg_final_score": np.mean(metrics["final_score"]),
        "std_final_score": np.std(metrics["final_score"])
    }

    return summary

def compute_accuracy(results, threshold=0.85):
    correct = sum(1 for r in results if r["semantic_score"] >= threshold)
    return correct / len(results)\

def score_distribution(results):
    scores = [r["final_score"] for r in results]

    return {
        "excellent (>0.9)": sum(1 for s in scores if s > 0.9),
        "good (0.8-0.9)": sum(1 for s in scores if 0.8 <= s <= 0.9),
        "poor (<0.8)": sum(1 for s in scores if s < 0.8)
    }

def get_worst_samples(dataset, results, n=5):
    paired = list(zip(dataset, results))
    sorted_samples = sorted(paired, key=lambda x: x[1]["final_score"])
    
    return sorted_samples[:n]

def evaluation(samples):
    all_results = []

    for sample in samples:
        res = evaluation_per_sample(sample, k=10)
        all_results.append(res)

    summary = aggregate_results(all_results)
    accuracy = compute_accuracy(all_results)
    distribution = score_distribution(all_results)
    # def convert(o):
    #     import numpy as np
    #     if isinstance(o, np.float32):
    #         return float(o)
    #     raise TypeError

    # with open("test.json","w") as f:
    #     json.dump(all_results, f, indent=4, default=convert)

    return {
        "summary":summary,
        "accuracy":accuracy,
        "distribution":distribution
    }

# MPNET EMBEDDING

## PHI3_MINI

In [10]:
LLM_MODEL = "microsoft/phi-3-mini-4k-instruct"
print("Loading LLM...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    dtype=torch.float16,
    device_map="auto",
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)

Loading LLM...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [13]:
dataset = test_data_creation(model,tokenizer)

1/50


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2/50
3/50
4/50
5/50
6/50
7/50
8/50
9/50
10/50
11/50
12/50
13/50
14/50
15/50
16/50
17/50
18/50
19/50
20/50
21/50
22/50
23/50
24/50
25/50
26/50
27/50
28/50
29/50
30/50
31/50
32/50
33/50
34/50
35/50
36/50
37/50
38/50
39/50
40/50
41/50
42/50
43/50
44/50
45/50
46/50
47/50
48/50
49/50
50/50


In [20]:
evaluation(dataset)

{'summary': {'avg_semantic_score': np.float32(0.82117426),
  'avg_grounding_score': np.float64(0.5730309990001373),
  'recall_at_k': np.float64(0.86),
  'avg_precision_at_k': np.float64(0.4537142857142857),
  'avg_keyword_score': np.float64(0.6159772443839493),
  'avg_hallucination_score': np.float64(0.4774061429368708),
  'avg_final_score': np.float32(0.67699194),
  'std_final_score': np.float32(0.17683338)},
 'accuracy': 0.42,
 'distribution': {'excellent (>0.9)': 5,
  'good (0.8-0.9)': 7,
  'poor (<0.8)': 38}}

## MISTRAL

In [4]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00:00:0100:01


In [5]:
import bitsandbytes as bnb
print(bnb.__version__)

0.49.2


In [6]:
import bitsandbytes as bnb
import torch

print(torch.cuda.is_available())        # Must be True
print(torch.cuda.device_count())        # Should show your GPU count
print(bnb.cuda_specs)                   # Check what bnb sees

True
2
<module 'bitsandbytes.cuda_specs' from '/usr/local/lib/python3.12/dist-packages/bitsandbytes/cuda_specs.py'>


In [10]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# bnb_config = BitsAndBytesConfig(
#     load_in_8bit=True,
#     bnb_8bit_compute_dtype=torch.float16,
#     bnb_8bit_use_double_quant=True,
#     bnb_8bit_quant_type="nf4"
# )

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    # quantization_config=bnb_config,
    dtype=torch.float16,
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [11]:
dataset = test_data_creation(model,tokenizer)

1/50


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2/50
3/50
4/50
5/50
6/50
7/50
8/50
9/50
10/50
11/50
12/50
13/50
14/50
15/50
16/50
17/50
18/50
19/50
20/50
21/50
22/50
23/50
24/50
25/50
26/50
27/50
28/50
29/50
30/50
31/50
32/50
33/50
34/50
35/50
36/50
37/50
38/50
39/50
40/50
41/50
42/50
43/50
44/50
45/50
46/50
47/50
48/50
49/50
50/50


In [12]:
evaluation(dataset)

{'summary': {'avg_semantic_score': np.float32(0.8609579),
  'avg_grounding_score': np.float64(0.5730309990001373),
  'recall_at_k': np.float64(0.86),
  'avg_precision_at_k': np.float64(0.4537142857142857),
  'avg_keyword_score': np.float64(0.6231913823191961),
  'avg_hallucination_score': np.float64(0.4758676850517145),
  'avg_final_score': np.float32(0.68688893),
  'std_final_score': np.float32(0.17825295)},
 'accuracy': 0.66,
 'distribution': {'excellent (>0.9)': 3,
  'good (0.8-0.9)': 11,
  'poor (<0.8)': 36}}

## LLAMA

In [ ]:
from huggingface_hub import login
login("give your hugging face token")

In [10]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [11]:
dataset = test_data_creation(model,tokenizer)

1/50


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2/50
3/50
4/50
5/50
6/50
7/50
8/50
9/50
10/50
11/50
12/50
13/50
14/50
15/50
16/50
17/50
18/50
19/50
20/50
21/50
22/50
23/50
24/50
25/50
26/50
27/50
28/50
29/50
30/50
31/50
32/50
33/50
34/50
35/50
36/50
37/50
38/50
39/50
40/50
41/50
42/50
43/50
44/50
45/50
46/50
47/50
48/50
49/50
50/50


In [12]:
evaluation(dataset)

{'summary': {'avg_semantic_score': np.float32(0.82965416),
  'avg_grounding_score': np.float64(0.5730309990001373),
  'recall_at_k': np.float64(0.86),
  'avg_precision_at_k': np.float64(0.4537142857142857),
  'avg_keyword_score': np.float64(0.6198677982021595),
  'avg_hallucination_score': np.float64(0.38821858188826464),
  'avg_final_score': np.float32(0.67063886),
  'std_final_score': np.float32(0.16991705)},
 'accuracy': 0.4,
 'distribution': {'excellent (>0.9)': 0,
  'good (0.8-0.9)': 12,
  'poor (<0.8)': 38}}

## QWEN

In [8]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)

dataset = test_data_creation(model,tokenizer)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


... (3 outputs left out)

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

1/50


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2/50
3/50
4/50
5/50
6/50
7/50
8/50
9/50
10/50
11/50
12/50
13/50
14/50
15/50
16/50
17/50
18/50
19/50
20/50
21/50
22/50
23/50
24/50
25/50
26/50
27/50
28/50
29/50
30/50
31/50
32/50
33/50
34/50
35/50
36/50
37/50
38/50
39/50
40/50
41/50
42/50
43/50
44/50
45/50
46/50
47/50
48/50
49/50
50/50


In [9]:
evaluation(dataset)

{'summary': {'avg_semantic_score': np.float32(0.81358874),
  'avg_grounding_score': np.float64(0.5730309990001373),
  'recall_at_k': np.float64(0.86),
  'avg_precision_at_k': np.float64(0.4537142857142857),
  'avg_keyword_score': np.float64(0.6449263964519796),
  'avg_hallucination_score': np.float64(0.25588910005829624),
  'avg_final_score': np.float32(0.6583821),
  'std_final_score': np.float32(0.17042495)},
 'accuracy': 0.34,
 'distribution': {'excellent (>0.9)': 1,
  'good (0.8-0.9)': 10,
  'poor (<0.8)': 39}}

## DEEPSEEK

In [9]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype=torch.float16,
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)
dataset = test_data_creation(model,tokenizer)

config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

1/50


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2/50
3/50
4/50
5/50
6/50
7/50
8/50
9/50
10/50
11/50
12/50
13/50
14/50
15/50
16/50
17/50
18/50
19/50
20/50
21/50
22/50
23/50
24/50
25/50
26/50
27/50
28/50
29/50
30/50
31/50
32/50
33/50
34/50
35/50
36/50
37/50
38/50
39/50
40/50
41/50
42/50
43/50
44/50
45/50
46/50
47/50
48/50
49/50
50/50


In [10]:
evaluation(dataset)

{'summary': {'avg_semantic_score': np.float32(0.7970464),
  'avg_grounding_score': np.float64(0.5730309990001373),
  'recall_at_k': np.float64(0.86),
  'avg_precision_at_k': np.float64(0.4537142857142857),
  'avg_keyword_score': np.float64(0.6220778825035967),
  'avg_hallucination_score': np.float64(0.4009732183525977),
  'avg_final_score': np.float32(0.66513854),
  'std_final_score': np.float32(0.16956107)},
 'accuracy': 0.5,
 'distribution': {'excellent (>0.9)': 2,
  'good (0.8-0.9)': 9,
  'poor (<0.8)': 39}}

# BGE EMBEDDING

## PHI3_MINI

In [9]:
LLM_MODEL = "microsoft/phi-3-mini-4k-instruct"
print("Loading LLM...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    dtype=torch.float16,
    device_map="auto",
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)
dataset = test_data_creation(model,tokenizer)

Loading LLM...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

... (4 outputs left out)

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

1/50


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2/50
3/50
4/50
5/50
6/50
7/50
8/50
9/50
10/50
11/50
12/50
13/50
14/50
15/50
16/50
17/50
18/50
19/50
20/50
21/50
22/50
23/50
24/50
25/50
26/50
27/50
28/50
29/50
30/50
31/50
32/50
33/50
34/50
35/50
36/50
37/50
38/50
39/50
40/50
41/50
42/50
43/50
44/50
45/50
46/50
47/50
48/50
49/50
50/50


In [10]:
evaluation(dataset)

{'summary': {'avg_semantic_score': np.float32(0.87422216),
  'avg_grounding_score': np.float64(0.5859736317685085),
  'recall_at_k': np.float64(0.78),
  'avg_precision_at_k': np.float64(0.45885714285714285),
  'avg_keyword_score': np.float64(0.6515274192143569),
  'avg_hallucination_score': np.float64(0.5468835498020882),
  'avg_final_score': np.float32(0.6872532),
  'std_final_score': np.float32(0.18864165)},
 'accuracy': 0.62,
 'distribution': {'excellent (>0.9)': 6,
  'good (0.8-0.9)': 9,
  'poor (<0.8)': 35}}

## QWEN

In [8]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)

dataset = test_data_creation(model,tokenizer)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


... (3 outputs left out)

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

1/50


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2/50
3/50
4/50
5/50
6/50
7/50
8/50
9/50
10/50
11/50
12/50
13/50
14/50
15/50
16/50
17/50
18/50
19/50
20/50
21/50
22/50
23/50
24/50
25/50
26/50
27/50
28/50
29/50
30/50
31/50
32/50
33/50
34/50
35/50
36/50
37/50
38/50
39/50
40/50
41/50
42/50
43/50
44/50
45/50
46/50
47/50
48/50
49/50
50/50


In [9]:
evaluation(dataset)

{'summary': {'avg_semantic_score': np.float32(0.8585728),
  'avg_grounding_score': np.float64(0.5859736317685085),
  'recall_at_k': np.float64(0.78),
  'avg_precision_at_k': np.float64(0.45885714285714285),
  'avg_keyword_score': np.float64(0.6843384258918729),
  'avg_hallucination_score': np.float64(0.25550258438201917),
  'avg_final_score': np.float32(0.66057813),
  'std_final_score': np.float32(0.18545437)},
 'accuracy': 0.66,
 'distribution': {'excellent (>0.9)': 0,
  'good (0.8-0.9)': 15,
  'poor (<0.8)': 35}}

## LLAMA

In [ ]:
from huggingface_hub import login
login("your token hugging face")

model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)

dataset = test_data_creation(model,tokenizer)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

... (2 outputs left out)

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

1/50


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2/50
3/50
4/50
5/50
6/50
7/50
8/50
9/50
10/50
11/50
12/50
13/50
14/50
15/50
16/50
17/50
18/50
19/50
20/50
21/50
22/50
23/50
24/50
25/50
26/50
27/50
28/50
29/50
30/50
31/50
32/50
33/50
34/50
35/50
36/50
37/50
38/50
39/50
40/50
41/50
42/50
43/50
44/50
45/50
46/50
47/50
48/50
49/50
50/50


In [9]:
evaluation(dataset)

{'summary': {'avg_semantic_score': np.float32(0.85959166),
  'avg_grounding_score': np.float64(0.5859736317685085),
  'recall_at_k': np.float64(0.78),
  'avg_precision_at_k': np.float64(0.45885714285714285),
  'avg_keyword_score': np.float64(0.6809886379579992),
  'avg_hallucination_score': np.float64(0.4494896798756743),
  'avg_final_score': np.float32(0.6795981),
  'std_final_score': np.float32(0.18777698)},
 'accuracy': 0.54,
 'distribution': {'excellent (>0.9)': 0,
  'good (0.8-0.9)': 16,
  'poor (<0.8)': 34}}

## MISTRAL

In [8]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    # quantization_config=bnb_config,
    dtype=torch.float16,
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)
dataset = test_data_creation(model,tokenizer)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

... (2 outputs left out)

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

1/50


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2/50
3/50
4/50
5/50
6/50
7/50
8/50
9/50
10/50
11/50
12/50
13/50
14/50
15/50
16/50
17/50
18/50
19/50
20/50
21/50
22/50
23/50
24/50
25/50
26/50
27/50
28/50
29/50
30/50
31/50
32/50
33/50
34/50
35/50
36/50
37/50
38/50
39/50
40/50
41/50
42/50
43/50
44/50
45/50
46/50
47/50
48/50
49/50
50/50


In [9]:
evaluation(dataset)

{'summary': {'avg_semantic_score': np.float32(0.90454584),
  'avg_grounding_score': np.float64(0.5859736317685085),
  'recall_at_k': np.float64(0.78),
  'avg_precision_at_k': np.float64(0.45885714285714285),
  'avg_keyword_score': np.float64(0.6822260400517574),
  'avg_hallucination_score': np.float64(0.5525187663875485),
  'avg_final_score': np.float32(0.7000136),
  'std_final_score': np.float32(0.1946225)},
 'accuracy': 0.86,
 'distribution': {'excellent (>0.9)': 7,
  'good (0.8-0.9)': 12,
  'poor (<0.8)': 31}}

## DEEPSEEK

In [8]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype=torch.float16,
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)
dataset = test_data_creation(model,tokenizer)

config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

1/50


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2/50
3/50
4/50
5/50
6/50
7/50
8/50
9/50
10/50
11/50
12/50
13/50
14/50
15/50
16/50
17/50
18/50
19/50
20/50
21/50
22/50
23/50
24/50
25/50
26/50
27/50
28/50
29/50
30/50
31/50
32/50
33/50
34/50
35/50
36/50
37/50
38/50
39/50
40/50
41/50
42/50
43/50
44/50
45/50
46/50
47/50
48/50
49/50
50/50


In [9]:
evaluation(dataset)

{'summary': {'avg_semantic_score': np.float32(0.86712873),
  'avg_grounding_score': np.float64(0.5859736317685085),
  'recall_at_k': np.float64(0.78),
  'avg_precision_at_k': np.float64(0.45885714285714285),
  'avg_keyword_score': np.float64(0.669340942786377),
  'avg_hallucination_score': np.float64(0.3889031993575394),
  'avg_final_score': np.float32(0.67310095),
  'std_final_score': np.float32(0.19226925)},
 'accuracy': 0.64,
 'distribution': {'excellent (>0.9)': 2,
  'good (0.8-0.9)': 14,
  'poor (<0.8)': 34}}

# e5-large-v2

## PHI3-MINI

In [8]:
LLM_MODEL = "microsoft/phi-3-mini-4k-instruct"
print("Loading LLM...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    dtype=torch.float16,
    device_map="auto",
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)
dataset = test_data_creation(model,tokenizer)

Loading LLM...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]


Processing:   0%|          | 0/50 [00:00<?, ?question/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Processing: 100%|██████████| 50/50 [11:49<00:00, 14.19s/question]


In [9]:
evaluation(dataset)

{'summary': {'avg_semantic_score': np.float32(0.9319444),
  'avg_grounding_score': np.float64(0.611418777243067),
  'recall_at_k': np.float64(0.82),
  'avg_precision_at_k': np.float64(0.4446904761904762),
  'avg_keyword_score': np.float64(0.668755763339847),
  'avg_hallucination_score': np.float64(0.5209294082684519),
  'avg_final_score': np.float32(0.7124212),
  'std_final_score': np.float32(0.16978647)},
 'accuracy': 1.0,
 'distribution': {'excellent (>0.9)': 5,
  'good (0.8-0.9)': 12,
  'poor (<0.8)': 33}}

## MISTRAL

In [8]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    # quantization_config=bnb_config,
    dtype=torch.float16,
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)
dataset = test_data_creation(model,tokenizer)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]


Processing:   0%|          | 0/50 [00:00<?, ?question/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Processing: 100%|██████████| 50/50 [08:01<00:00,  9.63s/question]


In [9]:
evaluation(dataset)

{'summary': {'avg_semantic_score': np.float32(0.9515198),
  'avg_grounding_score': np.float64(0.611418777243067),
  'recall_at_k': np.float64(0.82),
  'avg_precision_at_k': np.float64(0.4446904761904762),
  'avg_keyword_score': np.float64(0.7029884529315019),
  'avg_hallucination_score': np.float64(0.5548366199019206),
  'avg_final_score': np.float32(0.7262804),
  'std_final_score': np.float32(0.16845118)},
 'accuracy': 1.0,
 'distribution': {'excellent (>0.9)': 6,
  'good (0.8-0.9)': 13,
  'poor (<0.8)': 31}}

## LLAMA

In [ ]:
from huggingface_hub import login
login("your token")

model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)

dataset = test_data_creation(model,tokenizer)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Processing: 100%|██████████| 50/50 [21:05<00:00, 25.31s/question]


In [9]:
evaluation(dataset)

{'summary': {'avg_semantic_score': np.float32(0.91971666),
  'avg_grounding_score': np.float64(0.611418777243067),
  'recall_at_k': np.float64(0.82),
  'avg_precision_at_k': np.float64(0.4446904761904762),
  'avg_keyword_score': np.float64(0.6923521797886842),
  'avg_hallucination_score': np.float64(0.48867865136743044),
  'avg_final_score': np.float32(0.7107534),
  'std_final_score': np.float32(0.17356886)},
 'accuracy': 0.98,
 'distribution': {'excellent (>0.9)': 3,
  'good (0.8-0.9)': 13,
  'poor (<0.8)': 34}}

## QWEN

In [8]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True,
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)

dataset = test_data_creation(model,tokenizer)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]


Processing:   0%|          | 0/50 [00:00<?, ?question/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Processing: 100%|██████████| 50/50 [19:11<00:00, 23.04s/question]


In [9]:
evaluation(dataset)

{'summary': {'avg_semantic_score': np.float32(0.9222856),
  'avg_grounding_score': np.float64(0.611418777243067),
  'recall_at_k': np.float64(0.82),
  'avg_precision_at_k': np.float64(0.4446904761904762),
  'avg_keyword_score': np.float64(0.7168736338106689),
  'avg_hallucination_score': np.float64(0.2840471849647427),
  'avg_final_score': np.float32(0.6952692),
  'std_final_score': np.float32(0.16886073)},
 'accuracy': 1.0,
 'distribution': {'excellent (>0.9)': 2,
  'good (0.8-0.9)': 12,
  'poor (<0.8)': 36}}

## DEEPSEEK

In [8]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype=torch.float16,
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)
dataset = test_data_creation(model,tokenizer)

config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]


Processing:   0%|          | 0/50 [00:00<?, ?question/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Processing: 100%|██████████| 50/50 [13:55<00:00, 16.71s/question]


In [9]:
evaluation(dataset)

{'summary': {'avg_semantic_score': np.float32(0.9285935),
  'avg_grounding_score': np.float64(0.611418777243067),
  'recall_at_k': np.float64(0.82),
  'avg_precision_at_k': np.float64(0.4446904761904762),
  'avg_keyword_score': np.float64(0.6845293176527671),
  'avg_hallucination_score': np.float64(0.42009937648468293),
  'avg_final_score': np.float32(0.7044402),
  'std_final_score': np.float32(0.16737024)},
 'accuracy': 1.0,
 'distribution': {'excellent (>0.9)': 4,
  'good (0.8-0.9)': 13,
  'poor (<0.8)': 33}}

# API

In [1]:
!pip install faiss-cpu sentence-transformers transformers torch numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.9 MB/s eta 0:00:00:00:0100:01


In [2]:
import faiss
import json
import pickle
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import Dataset
import pandas as pd
from tqdm import tqdm

In [3]:
# ---------- CONFIG ----------
INDEX_PATH = "/kaggle/input/datasets/arjunsreenivasiiitk/justly/e5_large_index.faiss"
META_PATH = "/kaggle/input/datasets/arjunsreenivasiiitk/justly/e5_large_index.pkl"
EMBED_MODEL = "intfloat/e5-large-v2"
TOP_K = 10
# ----------------------------

print("Loading index...")
index = faiss.read_index(INDEX_PATH)

with open(META_PATH, "rb") as f:
    metadatas = pickle.load(f)

print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)

Loading index...
Loading embedding model...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

... (3 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [4]:
def retrieve(query, k=TOP_K):
    q_emb = embedder.encode(
        query,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(np.array([q_emb]), k)

    results = []
    for idx in indices[0]:
        results.append(metadatas[idx])

    return results

def clean_text(text):
    bad_tokens = ["Question:", "Answer:", "Q:", "A:"]
    for t in bad_tokens:
        text = text.replace(t, "")
    return text.strip()


def build_context(docs, max_chars=3000):
    lines = []
    total = 0
    for d in docs:
        chunk = d["text"]
        if total + len(chunk) > max_chars:
            break
        meta = ", ".join(f"{k}: {v}" for k, v in d["metadata"].items())
        lines.append(f"- {chunk} ({meta})")
        total += len(chunk)
    return "\n".join(lines)

def handle_query(query,tokenizer,model):
    docs = retrieve(query)
    context = build_context(docs)

    prompt = f"""
      You are an expert Ayurvedic biomedical knowledge assistant.

      Rules:
      - Use ONLY the context
      - If information is missing, say "Not available in context"
      - Answer as a complete paragraph of all items found in context.
      - Do not stop early.
      - If the user asks for a list, give output in a list.
      - Combine all the context text and give answer in a simple paragraph.



      Context:
        {context}

        Question:
        {query}

        Answer:
      """


    # print("tokenization started")
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    # print("tokenization ended")

    # print("generation started")
    output = model.generate(
      **inputs,
      max_new_tokens=300,     # ↓ from 300
      temperature=0.2,
      do_sample=False,        # IMPORTANT
      eos_token_id=tokenizer.eos_token_id,
      pad_token_id=tokenizer.eos_token_id,
      use_cache=True
    )

    # print("generation ended")

    # 🔑 decode ONLY new tokens
    generated = output[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)

    if "Context:" in text:
        text = text.split("Context:")[0]
    return text.strip(),context

In [5]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    # quantization_config=bnb_config,
    dtype=torch.float16,
    max_memory={
        0: "15GiB",  
        1: "15GiB",  
        "cpu": "30GiB"
    }
)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [6]:
!pip install fastapi uvicorn pyngrok nest-asyncio

In [20]:
import nest_asyncio
import threading
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn

nest_asyncio.apply()

app = FastAPI()

class PredictRequest(BaseModel):
    query: str

@app.get("/")
def home():
    return {"message": "Kaggle API is Live!"}

@app.post("/predict")
def predict(request: PredictRequest):
    try:
        user_query = request.query
        result = handle_query(user_query, tokenizer, model)
        return {"status": "success", "received": result}
    except Exception as e:
        return {"status": "error", "message": str(e)}

# 🔥 Run server in background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server)
thread.start()

# 🔥 Start ngrok AFTER server starts
ngrok.set_auth_token("YOUR_TOKEN")
public_url = ngrok.connect(8000)

print(f"🚀 API PUBLIC URL: {public_url}")

INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


🚀 API PUBLIC URL: NgrokTunnel: "https://lisabeth-floodable-fidgetingly.ngrok-free.dev" -> "http://localhost:8000"
INFO:     117.250.240.128:0 - "GET / HTTP/1.1" 200 OK
INFO:     117.250.240.128:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     117.250.240.128:0 - "GET /openapi.json HTTP/1.1" 200 OK


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


INFO:     117.250.240.128:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     117.250.240.128:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     117.250.240.128:0 - "POST /predict HTTP/1.1" 200 OK


In [19]:
import threading

# 3. Define a function to start the server
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# 4. Start the server in a background thread
# This keeps the Colab cell "active" without crashing the loop
thread = threading.Thread(target=run_server)
thread.start()

print(f"Server is running in the background!")
print(f"Check your API here: {public_url}")

Server is running in the background!
Check your API here: NgrokTunnel: "https://lisabeth-floodable-fidgetingly.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [16]:
from pyngrok import ngrok

# 1. Disconnect and kill all active tunnels
ngrok.kill()
print("Ngrok tunnel closed.")

# 2. To stop the uvicorn server thread, the easiest way in Colab
# is to clear the port or restart the runtime, but you can also do:
import os
import signal

# Note: This will restart the Colab backend, which is the cleanest way
# to ensure the port 8000 is released for a fresh run.
os.kill(os.getpid(), signal.SIGINT)

Ngrok tunnel closed.


KeyboardInterrupt: 

# END